# Module 2 — Lab

> Three constraints were written in the lesson, and every one of them had a
> number in it that somebody chose. This lab is about those numbers.

The graph is given, built by the same mapping you watched. What is not given is
the judgement, and each exercise asks for the same two things: **a count, and
one sentence defending it.**

| | the number somebody chose | what it decides |
|---|---|---|
| 1 | the upper bound on a contract's dates | whether an end date in 2104 is a typo or a covenant |
| 2 | how good a name must be before a shared postcode counts | which spellings become one organisation |
| 3 | nothing — the question is what the graph answers | whether a supplier was licensed when the city signed |

Nothing here needs a server. Same two CSV samples, same two vocabularies.

In [ ]:
# Pinned to the patch, for the reason module 1 gives: twenty people start this
# stack on different days, and a version left to float works for whoever wrote
# the notebook and fails for whoever runs it.
# pandas comes from the environment. No owlrl: the lab does not reason.
!pip install -q --disable-pip-version-check rdflib==7.6.0 pyshacl==0.40.1 kuzu==0.11.3 certifi==2025.8.3

# Everything the lesson built, in one cell. Reading it is not the exercise: it
# is the same mapping, the same shapes helper and the same similarity rules,
# condensed so that the three questions below have something to run against.
import datetime, re
from itertools import combinations
from pathlib import Path

import pandas as pd
from pyshacl import validate
from rdflib import Graph, Literal, Namespace, RDF, XSD, URIRef
from rdflib.namespace import DCTERMS

pd.set_option("display.max_colwidth", None)

CONTRACT_ID = "Purchase Order (Contract) Number"
PC, GR, KGTP = ("http://purl.org/procurement/public-contracts#",
                "http://purl.org/goodrelations/v1#", "http://kgtp.org/")
PCO, GRO, KG, SH = Namespace(PC), Namespace(GR), Namespace(KGTP), Namespace("http://www.w3.org/ns/shacl#")

# Four places this can run from: the module root (the lab), the
# solutions folder one level down, a checkout of the course repository, and
# Colab. The first three are directories; the fourth is a URL.
RAW = ("https://raw.githubusercontent.com/giuseppefutia/kgtp-course/"
       "main/modules/02-modeling-the-world/")

def asset(name):
    for candidate in (Path(name), Path("..") / name, Path("../..") / name):
        if candidate.exists():
            return str(candidate)
    return RAW + name

DATA = ""

contracts = pd.read_csv(asset("data/contracts_sample.csv"), dtype=str).fillna("")
licences = pd.read_csv(asset("data/licenses_sample.csv"), dtype=str).fillna("")

US_DATE = re.compile(r"^(\d{1,2})/(\d{1,2})/(\d{1,4})$")

def as_date(value):
    match = US_DATE.match(value.strip())
    if not match:
        return None
    month, day, year = (int(part) for part in match.groups())
    try:
        return Literal(datetime.date(year, month, day), datatype=XSD.date)
    except ValueError:
        return None

to_date = lambda v: (as_date(v).toPython() if as_date(v) is not None else None)
slug = lambda text: re.sub(r"[^A-Za-z0-9]+", "-", text).strip("-").lower()

g = Graph()
for prefix, uri in [("pc", PCO), ("gr", GRO), ("kgtp", KG), ("dcterms", DCTERMS)]:
    g.bind(prefix, uri)

for contract_id, rows in contracts.groupby(CONTRACT_ID):
    contract = KG[f"contract-{contract_id}"]
    g.add((contract, RDF.type, PCO.Contract))
    g.add((contract, DCTERMS.identifier, Literal(contract_id)))
    for _, row in rows.iterrows():
        record = KG[f"record-{contract_id}-{row['Revision Number']}"]
        g.add((record, RDF.type, KG.ContractRecord))
        g.add((record, KG.revisionOf, contract))
        g.add((record, KG.revisionNumber, Literal(row["Revision Number"])))
        for column, prop in [("Approval Date", PCO.awardDate), ("Start Date", PCO.startDate),
                             ("End Date", PCO.actualEndDate)]:
            typed = as_date(row[column])
            if typed is not None:
                g.add((record, prop, typed))
        if row["Department"]:
            g.add((record, PCO.contractingAuthority, KG[f"dept-{slug(row['Department'])}"]))
    for name in sorted({r["Department"] for _, r in rows.iterrows() if r["Department"]}):
        department = KG[f"dept-{slug(name)}"]
        g.add((contract, PCO.contractingAuthority, department))
        g.add((department, RDF.type, GRO.BusinessEntity))
        g.add((department, GRO.legalName, Literal(name)))
    awarded = sorted(d for d in (as_date(v) for v in rows["Approval Date"]) if d is not None)
    if awarded:
        g.add((contract, PCO.awardDate, awarded[0]))

def check(shapes_turtle, data=None):
    """Validate the data graph against one shapes graph; return (conforms, report)."""
    shapes = Graph()
    shapes.parse(data=shapes_turtle, format="turtle")
    conforms, report, _ = validate(data if data is not None else g, shacl_graph=shapes,
                                   inference="none", advanced=True)
    return conforms, report

def violations(report):
    """The report as rows, so an exercise can count and read them."""
    out = []
    for result in report.subjects(RDF.type, SH.ValidationResult):
        out.append((str(report.value(result, SH.focusNode)).replace(KGTP, "kgtp:"),
                    str(report.value(result, SH.resultPath)).split("#")[-1],
                    str(report.value(result, SH.value) or ""),
                    str(report.value(result, SH.resultMessage))))
    return sorted(out)

LEGAL_FORM = re.compile(r"\b(?:co|ltd|inc|corp|llc|llp|nfp|company|incorporated)\b", re.I)
UNIT = re.compile(r"\b(?:ste|suite|apt|unit|fl|floor|#)\b.*$|\s+\d+\s+\d+$|\s+[A-Z]?\d{2,4}$", re.I)

def name_similarity(left, right):
    def bigrams(name):
        cleaned = re.sub(r"[^a-z0-9]+", "", LEGAL_FORM.sub(" ", name.lower()))
        return {cleaned[i:i + 2] for i in range(len(cleaned) - 1)}
    a, b = bigrams(left), bigrams(right)
    return 2 * len(a & b) / (len(a) + len(b)) if a and b else 0.0

without_unit = lambda address: re.sub(r"\s+", " ", UNIT.sub("", address.upper())).strip()

parties = {}
for _, row in contracts.drop_duplicates("Vendor ID").iterrows():
    parties[row["Vendor Name"]] = ("contracts", {row["Address 1"].upper()}, {row["Zip"]})
for name, rows in licences.groupby("LEGAL NAME"):
    parties[name] = ("licences", set(rows["ADDRESS"].str.upper()), set(rows["ZIP CODE"]))

def propose_merges(rules):
    """Run the resolution rules over the parties and return what they join."""
    found = []
    for left, right in combinations(parties, 2):
        (_, addresses_l, zips_l), (_, addresses_r, zips_r) = parties[left], parties[right]
        score = name_similarity(left, right)
        methods = []
        for label, floor, matches in rules:
            shares = matches(addresses_l, addresses_r) if matches else bool(zips_l & zips_r)
            if shares and score >= floor:
                methods.append(label)
        if methods:
            found.append((round(score, 2), methods, left, right))
    return sorted(found, reverse=True)

RULES = [("SAME_ADDRESS", 0.3, lambda a, b: bool(a & b)),
         ("NORM_ADDRESS", 0.5, lambda a, b: bool({without_unit(x) for x in a} &
                                                 {without_unit(x) for x in b})),
         ("SAME_ZIP",     0.8, None)]

# This cell is a second copy of the lesson's mapping, and two copies drift. The
# dangerous way for them to drift is quietly: the lab would keep running and
# answer a slightly different question, with nothing to show that it had.
#
# So the three facts the exercises below actually rest on are asserted here. If
# the lesson changes shape, this stops rather than lies.
EXPECTED = {"contracts": 23, "revisions": 46, "spellings": 15, "merges": 2}
actual = {"contracts": len(set(g.subjects(RDF.type, PCO.Contract))),
          "revisions": len(set(g.subjects(RDF.type, KG.ContractRecord))),
          "spellings": len(parties),
          "merges": len(propose_merges(RULES))}
if actual != EXPECTED:
    raise AssertionError(
        f"the sample or the mapping has changed: expected {EXPECTED}, got {actual}. "
        "The answers below were worked out on the old shape — rebuild them before trusting them.")

print(f"{len(g)} statements, {actual['contracts']} contracts, {actual['revisions']} revisions, "
      f"{actual['spellings']} party spellings.")

## 1. The bound you had to defend

The lesson bounded start and end dates between **1900** and **2100**, and the
report named eight revisions. Seven are plainly typing errors — years 201, 202,
204, 207, and one contract running to 3030. The eighth is different:

    kgtp:record-14775-0   actualEndDate   2104-01-24

Seventy-eight years out. A slip for 2014, or a very long land covenant? Nothing
in the data decides it, and **the bound you pick decides it for you.**

Write the same two shapes with the upper bound moved to **2110-12-31**, run
them, and count.

**Produce:** the number of violations at each bound, and one sentence saying
which bound you would ship and why. There is no right answer here; there is a
defended one and an undefended one.

## 2. The merge the threshold refused

The lesson set out the three signals and what each demands of the name, and
showed the pair these rules decline: `ROSELAND CHRISTIAN HEALTH M 01` and
`ROSELAND CHRISTIAN MINISTRIES CENTER`, the same body, scoring **0.58** with
only a shared postcode to go on — and a postcode demands 0.8.

Lower the `SAME_ZIP` floor to **0.5** and re-run `propose_merges`.

**Produce:** what the loosened rule proposes that the strict one did not, which
of those you would accept, and one sentence on why the same change would be
dangerous on the full dataset rather than on fifteen spellings.

## 3. The answer you did not ask for

The last act walked contract → supplier → resolved identity → licence term, and
counted the approvals signed **after** the supplier's last licence expired: 477
of them, from 2015 to 2026.

That number only means something next to the one nobody printed.

The cell below rebuilds that graph — it is the lesson's, given. Then **ask it the
complement**: how many approvals fell *inside* a licence term, and when.

**Produce:** the two counts side by side, and one sentence on what the pair says
that either number alone does not.

In [ ]:
import kuzu, os, tempfile

# The integration graph from the lesson, rebuilt. Given, not an exercise.
integration_contracts = pd.read_csv(asset("data/integration_contracts.csv"), dtype=str).fillna("")
integration_licences = pd.read_csv(asset("data/integration_licenses.csv"), dtype=str).fillna("")

ink = kuzu.Connection(kuzu.Database(os.path.join(tempfile.mkdtemp(), "graph")))
for statement in [
    "CREATE NODE TABLE Party(id STRING, name STRING, source STRING, PRIMARY KEY(id))",
    "CREATE NODE TABLE Contract(id STRING, PRIMARY KEY(id))",
    "CREATE NODE TABLE Revision(id STRING, approvedOn DATE, PRIMARY KEY(id))",
    "CREATE NODE TABLE LicenceTerm(id STRING, validFrom DATE, validUntil DATE, PRIMARY KEY(id))",
    "CREATE REL TABLE AMENDS(FROM Revision TO Contract)",
    "CREATE REL TABLE AWARDED_TO(FROM Contract TO Party)",
    "CREATE REL TABLE HOLDS(FROM Party TO LicenceTerm)",
    "CREATE REL TABLE RESOLVED_WITH(FROM Party TO Party, score DOUBLE, methods STRING[])",
]:
    ink.execute(statement)

sides = {}

def party(source, name, address, postcode, identifier):
    """Same as the lesson's, identifiers included — two copies that differ are
    worse than two copies that agree."""
    key = f"{source}:{name}"
    if key not in sides:
        sides[key] = (source, name, {address.upper()}, {postcode}, {identifier})
        ink.execute("CREATE (:Party {id: $id, name: $name, source: $source})",
                    {"id": key, "name": name, "source": source})
    else:
        sides[key][2].add(address.upper())
        sides[key][3].add(postcode)
        sides[key][4].add(identifier)
    return key

for _, row in integration_contracts.iterrows():
    contract_id = row[CONTRACT_ID]
    ink.execute("MERGE (:Contract {id: $id})", {"id": contract_id})
    approved = to_date(row["Approval Date"])
    if approved is not None:
        revision = f"{contract_id}-{row['Revision Number']}"
        ink.execute("""MERGE (r:Revision {id: $id})
                       ON CREATE SET r.approvedOn = $approved""",
                    {"id": revision, "approved": approved})
        ink.execute("""MATCH (r:Revision {id: $rev}), (c:Contract {id: $id})
                       MERGE (r)-[:AMENDS]->(c)""", {"rev": revision, "id": contract_id})
    key = party("contracts", row["Vendor Name"], row["Address 1"], row["Zip"],
                row["Vendor ID"])
    ink.execute("""MATCH (c:Contract {id: $id}), (p:Party {id: $party})
                   MERGE (c)-[:AWARDED_TO]->(p)""", {"id": contract_id, "party": key})

for _, row in integration_licences.iterrows():
    key = party("licences", row["LEGAL NAME"], row["ADDRESS"], row["ZIP CODE"],
                row["ACCOUNT NUMBER"])
    start, until = to_date(row["LICENSE TERM START DATE"]), to_date(row["LICENSE TERM EXPIRATION DATE"])
    if start is None or until is None:
        continue
    ink.execute("""MERGE (l:LicenceTerm {id: $id})
                   ON CREATE SET l.validFrom = $since, l.validUntil = $until""",
                {"id": row["ID"], "since": start, "until": until})
    ink.execute("""MATCH (p:Party {id: $party}), (l:LicenceTerm {id: $id})
                   MERGE (p)-[:HOLDS]->(l)""", {"party": key, "id": row["ID"]})

for left, right in combinations(sides, 2):
    (_, name_l, addresses_l, zips_l, _) = sides[left]
    (_, name_r, addresses_r, zips_r, _) = sides[right]
    score = name_similarity(name_l, name_r)
    methods = []
    for label, floor, matches in RULES:
        shares = matches(addresses_l, addresses_r) if matches else bool(zips_l & zips_r)
        if shares and score >= floor:
            methods.append(label)
    if methods:
        ink.execute("""MATCH (a:Party {id: $left}), (b:Party {id: $right})
                       CREATE (a)-[:RESOLVED_WITH {score: $score, methods: $methods}]->(b)""",
                    {"left": left, "right": right, "score": score, "methods": methods})

print(f"{len(sides)} parties, {len(integration_contracts)} contract rows, "
      f"{len(integration_licences)} licence terms.")

## 4. The mapping somebody else would have written

The lesson mapped nineteen columns onto somebody else's vocabulary by hand, and
said that every line of it was a decision. Test that claim against a machine.

Give an LLM of your choice the header of `contracts_sample.csv`, five rows, and
this instruction:

> *Map these columns onto the Public Contracts Ontology (PCO) and write the
> result as Turtle. Say which term you chose for each column.*

Do not ask it whether it is right. Ask it to commit, then read what it committed.

**Produce:** three decisions it made that differ from ours, and for each one a
sentence saying which of the two you would ship. Not «the LLM was wrong» — a
difference, and a choice.

## 5. A vocabulary that fits better

PCO was published by a Czech procurement project in 2015, and the lesson showed
where it stops: no word for a revision, none for a business licence, none for the
fact that a supplier was the counterparty only between two dates.

So find a better one. Candidates worth half an hour: the EU **eProcurement
Ontology (ePO)**, which is maintained and current; **schema.org**, which has
`GovernmentPermit`; the W3C **Organization** ontology; **PROV-O**.

**Produce:** one candidate, three of its terms checked against three of our
columns — term, column, and whether the domain and range actually fit — and one
sentence on what it has that PCO lacks. Verify the terms exist by loading the
vocabulary, the way the lesson's mapping cell does. A term that reads plausibly
and is not there is the failure that cell guards against.

In [ ]:
# Load a candidate vocabulary and check the terms you picked are really in it.
# This is the lesson's guard, reused: a term that reads plausibly and is absent
# fails nowhere — the shapes simply never match anything.
#
# Two practicalities. The python.org build ships no CA bundle, so an https read
# fails with CERTIFICATE_VERIFY_FAILED and nothing about the message suggests
# the interpreter is the cause — hence certifi. And the candidate here is the
# W3C Organization ontology at 84 KB rather than schema.org at twelve megabytes:
# check the small one first, and only pay for the large one if it looks right.
import ssl, certifi, urllib.request
from rdflib import Graph, URIRef

ssl_context = ssl.create_default_context(cafile=certifi.where())
urllib.request.install_opener(
    urllib.request.build_opener(urllib.request.HTTPSHandler(context=ssl_context)))

CANDIDATE = "https://www.w3.org/ns/org.ttl"
TERMS = ["http://www.w3.org/ns/org#OrganizationalUnit",
         "http://www.w3.org/ns/org#unitOf",
         "http://www.w3.org/ns/org#ChangeEvent",
         "http://www.w3.org/ns/org#hasLicense"]      # this one is a guess — check it

candidate = Graph()
candidate.parse(CANDIDATE, format="turtle")
print(f"{len(candidate)} statements loaded from {CANDIDATE}\n")
for term in TERMS:
    present = (URIRef(term), None, None) in candidate
    print(f"   {'present' if present else 'ABSENT ':<8} {term}")


## What to hand in

Five answers. The first three are counts you can check; the last two are
differences you have to defend.

And one line, for yourself, on the exercise that was least comfortable: in all
five the machine did its part correctly and still could not finish the job. That
is not a shortcoming of the tools. It is where you come in.